# r2-001 Problem 2 — Reference solution (two-source heat localization)

One reference approach: a small learned inverse (MLP trained on training rows plus freshly simulated rows) gives each row an initial guess, and a short per-example least-squares refinement through the differentiable forward model polishes it.
The cell below is the problem notebook's supplied setup cell, unchanged.

In [ ]:
import importlib.util
import json
import math
import os
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261101
torch.set_num_threads(1)


def find_data_dir() -> Path:
    """Locate r2-001's data/ folder from the notebook's working directory."""
    candidates = [Path("../data"), Path("data"), Path("mocktests/r2-001/data"), Path("book2/mocktests/r2-001/data")]
    env_root = os.environ.get("USAAIO_BOOK_ROOT")
    if env_root:
        candidates.insert(0, Path(env_root) / "mocktests" / "r2-001" / "data")
    for path in candidates:
        if (path / "gen_r2_001.py").is_file():
            return path.resolve()
    raise FileNotFoundError("cannot find mocktests/r2-001/data; run from problems/ or set USAAIO_BOOK_ROOT")


DATA_DIR = find_data_dir()
_spec = importlib.util.spec_from_file_location("r2_001_data", DATA_DIR / "r2_001_data.py")
r2_001_data = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(r2_001_data)
assert r2_001_data.SEED == SEED

x_train, y_train = r2_001_data.train_rows("heat")
x_val, y_val = r2_001_data.val_rows("heat")
print(x_train.shape, y_train.shape, x_val.shape)
print("baseline recipe:", r2_001_data.BASELINE_RECIPES["heat"])

## Budget, forward-model check, and the committed baseline

In [ ]:
budget = r2_001_data.StepBudget(3000)

# Forward-model check: the noise-free readings explain the training rows to the stated noise level.
with torch.no_grad():
    y64 = y_train.double()
    clean = r2_001_data.heat_forward(y64[:, [0, 1, 3, 4]].reshape(-1, 2, 2), y64[:, [2, 5]])
    resid_sd = float((x_train.double() - clean).std())
print(f"training residual s.d. vs noise-free forward model: {resid_sd:.4f} (stated noise 0.05)")
assert abs(resid_sd - 0.05) < 0.005

baseline = r2_001_data.baseline_score("heat")
print("baseline under the protocol:", baseline)

## Approach: learned inverse + per-example least-squares refinement

In [ ]:
SIM_ROWS, HIDDEN, NET_STEPS, BATCH = 20000, 128, 1000, 512
REFINE_STEPS = 300

seam = set()
state = {}


def to_targets(y):
    """(N, 6) = (x1, y1, q1, x2, y2, q2) -> (x1, y1, x2, y2, log q1, log q2)."""
    return torch.stack([y[:, 0], y[:, 1], y[:, 3], y[:, 4], torch.log(y[:, 2]), torch.log(y[:, 5])], dim=1)


def pair_loss(pred, target):
    """Permutation-invariant squared error (positions, plus 0.1 x log-strengths)."""
    same = ((pred[:, :4] - target[:, :4]) ** 2).sum(1) + 0.1 * ((pred[:, 4:] - target[:, 4:]) ** 2).sum(1)
    swap = ((pred[:, [2, 3, 0, 1]] - target[:, :4]) ** 2).sum(1) + 0.1 * ((pred[:, [5, 4]] - target[:, 4:]) ** 2).sum(1)
    return torch.minimum(same, swap).mean()


def fit(train_x, train_y):
    for row in train_x:
        seam.add(r2_001_data.split_of("heat", row))
    mean, std = train_x.mean(0), train_x.std(0)
    torch.manual_seed(SEED)
    net = nn.Sequential(nn.Linear(50, HIDDEN), nn.GELU(), nn.Linear(HIDDEN, HIDDEN), nn.GELU(), nn.Linear(HIDDEN, 6))
    base_opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    sched = torch.optim.lr_scheduler.OneCycleLR(base_opt, max_lr=3e-3, total_steps=NET_STEPS)
    opt = budget.wrap(base_opt)
    xn, targets = (train_x - mean) / std, to_targets(train_y)
    gen = torch.Generator().manual_seed(SEED)
    for _ in range(NET_STEPS):
        idx = torch.randint(0, len(xn), (BATCH,), generator=gen)
        opt.zero_grad(set_to_none=True)
        loss = pair_loss(net(xn[idx]), targets[idx])
        loss.backward()
        opt.step()
        sched.step()
    state.update(net=net.eval(), mean=mean, std=std)
    print(f"learned inverse trained: {NET_STEPS} steps, last mini-batch loss {loss.item():.5f}")


def initial_guess(x):
    with torch.no_grad():
        out = state["net"]((x - state["mean"]) / state["std"])
    return out[:, :4].reshape(-1, 2, 2).double(), torch.exp(out[:, 4:]).double()


def predict(x, refine_steps=REFINE_STEPS, lr=None):
    xy, q = initial_guess(x)
    lr = state["refine_lr"] if lr is None else lr
    return r2_001_data.heat_ls_fit(x, xy, q, steps=refine_steps, lr=lr, optimizer_wrapper=budget.wrap)


x_sim, y_sim = r2_001_data.simulate("heat", SIM_ROWS, seed=7)
fit(torch.cat([x_train, x_sim]), torch.cat([y_train, y_sim]))
assert seam <= {"train", None} and "val" not in seam and "test" not in seam
print("seam:", seam, "| steps used after fit:", budget.used)

## Validation evidence (every choice is made here)

In [ ]:
def val_err(pred):
    return float(r2_001_data.pair_position_error_per_example(pred, y_val).mean())


xy0, q0 = initial_guess(x_val)
err_net_only = val_err(r2_001_data.canonical_heat(xy0).float())
xy_b, q_b = r2_001_data.heat_baseline_init(len(x_val))
err_fixed_start = val_err(r2_001_data.heat_ls_fit(x_val, xy_b, q_b, steps=300, lr=0.01, optimizer_wrapper=budget.wrap))
err_refine_lr01 = val_err(predict(x_val, lr=0.01))
err_refine_lr005 = val_err(predict(x_val, lr=0.005))
print(f"val error, learned inverse only (0 refinement steps):        {err_net_only:.5f}")
print(f"val error, fixed-start least squares (baseline recipe):      {err_fixed_start:.5f}")
print(f"val error, learned inverse + 300 refinement steps, lr 0.01:  {err_refine_lr01:.5f}")
print(f"val error, learned inverse + 300 refinement steps, lr 0.005: {err_refine_lr005:.5f}")
state["refine_lr"] = 0.01 if err_refine_lr01 <= err_refine_lr005 else 0.005
print("chosen refinement lr (lower val error):", state["refine_lr"])
print("steps used before the test call:", budget.used, "of", budget.max_steps)

## The single locked-test call

In [ ]:
assert r2_001_data.test_call_count("heat") == 0
result = r2_001_data.final_test_score(predict, n_boot=1000, alpha=0.05, task="heat")
assert r2_001_data.test_call_count("heat") == 1
elapsed = budget.elapsed()
print(result)
print(f"budget: {budget.used} of {budget.max_steps} steps, elapsed {elapsed:.1f} s")

## Writeup

**Approach.** The readings are an exactly known, differentiable function of the six unknowns (`heat_forward`), and with i.i.d. Gaussian noise the maximum-likelihood estimate is the per-row least-squares fit; its only weakness is the non-convex landscape, so the fixed-start baseline often lands in a wrong basin (two sources merged, or a source pulled toward the wrong sensor cluster).
I therefore split the work: a small learned inverse (MLP, 50 standardized readings → 128 → 128 → 6 outputs = two positions and two log-strengths, GELU, trained 1,000 Adam steps with a one-cycle schedule on the 800 training rows plus 20,000 rows from `simulate("heat", 20000, seed=7)` under a permutation-invariant squared loss) supplies a per-row starting point near the right basin, and 300 steps of per-row least squares (`heat_ls_fit`, Adam on positions and log-strengths) from that start polish it to the noise floor.
The early time $t_1=0.005$ (kernel s.d. $0.1$, half the sensor spacing) localizes each source; the later time $t_2=0.02$ spreads heat to more sensors and stabilizes the strengths.

**Alternatives considered.** (1) *Fixed-start least squares* (the baseline recipe, 300 steps, lr 0.01) on validation: mean error 0.03777, more than ten times the chosen pipeline's 0.00264, because many rows converge to a wrong basin.
(2) *Learned inverse alone* (no refinement): 0.03144 on validation — the amortized network with 1,000 CPU steps is a good initializer but not an accurate estimator.
(3) *Refinement learning rate 0.005 vs 0.01*: 0.00282 vs 0.00264 on validation, so lr 0.01 was selected programmatically.

**Evaluation.** All choices used the 200 validation rows only; the locked test rows were scored once with `final_test_score` (n = 200, 1,000 bootstrap resamples, α = 0.05).
Test score **0.00154** (mean pair-position error), 95% bootstrap interval [0.00083, 0.00287]; the committed baseline under the same protocol scores 0.04810 [0.03560, 0.06197], so the improvement is far outside both intervals and clears the full-credit threshold 0.01318.
The run used 2,200 of the 3,000 optimizer steps (1,000 network, 300 fixed-start alternative, 2 × 300 validation refinements, 300 test refinement) and finished in about 8.5 s by `budget.elapsed()` on this CPU.

**Limitations.** The mean is dominated by a few rows where the initializer points to the wrong basin (for example two sources near the 0.25 minimum separation, or a source near a corner where only one sensor sees it at $t_1$); refinement cannot escape such a basin.
I would test this by binning validation (or fresh simulated) errors by source separation and by distance to the nearest sensor, and by checking the residual norm of each fitted row: rows whose residual s.d. is well above 0.05 flag failed fits, which could then be retried from the swapped or a second initializer.
The approach also assumes the infinite-sheet forward model exactly; a plate with insulated edges would bias sources near the boundary, which a simulation with reflected image sources would reveal.

### Answer check

In [ ]:
assert r2_001_data.test_call_count("heat") == 1, "final_test_score must be called exactly once"
assert seam <= {"train", None}
assert budget.used <= 3000 and elapsed < 60
assert result.n_examples == 200 and result.ci_low <= result.score <= result.ci_high
assert result.score < baseline.score
assert result.score <= 0.01318, "full-credit tier: score <= R + 0.25 g"
print(f"Problem 2 test score: {result.score:.5f} (95% CI {result.ci_low:.5f}-{result.ci_high:.5f}, n = {result.n_examples}); "
      f"baseline {baseline.score:.5f}")